# Raw-data preprocessing for the equity research project

This notebook recreates the complete `data/preprocess` directory from files under `data/raw`. It produces audited native-currency financial schedules, USD-converted schedules, market-reference files, and the distribution case-study tables.

Every transformation is contained here. No preprocessing helper script is imported. Running all cells from a repository checkout with the raw files present is sufficient to rebuild the output directory.

## Sources and boundaries

- `data/raw/dcf_inputs/financial_facts.csv` contains SEC filing facts with source URLs and Markdown line references.
- `data/raw/dcf_inputs/filing_statement_rows.csv` preserves the table-level units used to validate scale.
- `data/raw/dcf_inputs/filing_index.csv` identifies the underlying SEC filings.
- `data/raw/dcf_inputs/wacc_market_inputs.csv` and `peer_market_snapshot.csv` contain reference market inputs.
- `data/raw/forex/chf-usd.csv` contains Yahoo Finance `CHFUSD=X` observations. The quote means USD per CHF.
- `data/raw/sec_filings` contains SEC filings converted to Markdown and is used for the channel case study.
- `data/raw/prices` contains Yahoo Finance daily adjusted closes for NKE, DECK, and ONON.

The raw SEC fact files are treated as supplied inputs. This notebook validates them but does not download filings or recreate `data/raw`.

## Manual choices and limitations

The following choices are manually specified in this notebook and are not inferred from a statistical model:

- NKE, DECK, and ONON are the DCF operating companies; Foot Locker is retained as contextual data.
- Historical model schedules use each company's latest three annual periods.
- DECK LTM uses the latest annual period plus the current three-month period minus the comparative three-month period.
- ONON LTM uses the latest annual period plus the current six-month period minus the comparative six-month period.
- NKE uses its latest annual period because no later Nike interim filing is present in the raw inputs.
- Fiscal year-end months are manually mapped as May for NKE, March for DECK, and December for ONON.
- ONON CHF amounts are translated using the latest valid adjusted CHF/USD close on or before the reporting date. This is a comparability translation, not a GAAP average-rate translation.
- The ONON price in the raw peer market snapshot is already a USD-listed share price and is not multiplied by CHF/USD.
- Channel-history source filings and table row labels are manually selected and documented near that extraction logic.
- Missing reported values remain missing. They are never silently replaced with zero.

## Rebuild verification

During development, the existing `data/preprocess` directory was manually deleted before this notebook was executed as a clean-room test. The first executable section also removes that directory every time the notebook runs, then recreates it. This deliberate destructive action applies only to generated preprocessing outputs.

## Imports and project paths

Only the Python standard library, NumPy, and pandas are required. The project root is located by searching for the raw financial-facts file, so the notebook works from the repository root or from its own directory.

In [ ]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/raw/dcf_inputs/financial_facts.csv").exists()
)
raw = root / "data/raw"
out = root / "data/preprocess"
case_out = out / "case_study"

## Reset the generated output directory

This deletion is intentional and manual policy encoded as executable code. Only `data/preprocess` is removed. The raw files, research notebooks, and model results are outside this target and are not touched.

In [ ]:
if out.exists():
    shutil.rmtree(out)

case_out.mkdir(parents=True, exist_ok=True)
print(f"Created empty output directory: {out}")

## Load the raw tables

Each input is loaded directly from `data/raw`. Dates are parsed immediately so comparisons cannot accidentally use string ordering.

In [ ]:
dcf_raw = raw / "dcf_inputs"
raw_facts = pd.read_csv(dcf_raw / "financial_facts.csv")
statement_rows = pd.read_csv(dcf_raw / "filing_statement_rows.csv")
filings = pd.read_csv(dcf_raw / "filing_index.csv")
wacc = pd.read_csv(dcf_raw / "wacc_market_inputs.csv")
market = pd.read_csv(dcf_raw / "peer_market_snapshot.csv")
fx_raw = pd.read_csv(raw / "forex/chf-usd.csv")

for frame in (raw_facts, filings, statement_rows):
    frame["period_end"] = pd.to_datetime(frame["period_end"])
    frame["filing_date"] = pd.to_datetime(frame["filing_date"])
fx_raw["date"] = pd.to_datetime(fx_raw["date"])

## Validate raw SEC provenance, units, and identities

ONON became public in 2021, but its first 20-F was filed in 2022 and includes audited comparative financial periods for 2019 through 2021. A 2019 `period_end` is therefore not a 2019 SEC filing.

That first ONON filing reports CHF in thousands, while later filings report CHF in millions. The assertions below explicitly guard this mixed-unit boundary and the known CHF 267.12 million 2019 revenue value. Nike share counts are also checked for plausible absolute units.

In [ ]:
fact_key = ["ticker", "period_end", "duration", "metric"]
assert not raw_facts.duplicated(fact_key).any(), "Duplicate raw SEC facts"
assert raw_facts["value"].notna().all()
assert np.isfinite(raw_facts["value"]).all()
assert raw_facts["filing_date"].ge(raw_facts["period_end"]).all()
assert raw_facts["source_line"].notna().all()
assert raw_facts["source_url"].str.startswith("https://www.sec.gov/Archives/").all()
assert raw_facts["source_url"].isin(filings["source_url"]).all()

In [ ]:
first_onon = statement_rows.loc[
    statement_rows.ticker.eq("ONON")
    & statement_rows.filing_date.eq("2022-03-18")
]
later_onon = statement_rows.loc[
    statement_rows.ticker.eq("ONON")
    & statement_rows.filing_date.gt("2022-03-18")
]
assert set(first_onon.reported_scale) == {1_000}
assert set(later_onon.reported_scale) == {1_000_000}

onon_2019_revenue = raw_facts.loc[
    raw_facts.ticker.eq("ONON")
    & raw_facts.period_end.eq("2019-12-31")
    & raw_facts.metric.eq("revenue"), "value"
]
assert onon_2019_revenue.tolist() == [267_120_000]

In [ ]:
nike_share_metrics = {
    "class_a_shares_outstanding", "class_b_shares_outstanding",
    "weighted_average_basic_shares", "weighted_average_diluted_shares",
}
nike_shares = raw_facts.loc[
    raw_facts.ticker.eq("NKE")
    & raw_facts.metric.isin(nike_share_metrics), "value"
]
assert nike_shares.between(100_000_000, 2_000_000_000).all()

annual_identity = raw_facts.loc[raw_facts.duration.eq("annual")].pivot(
    index=["ticker", "period_end"], columns="metric", values="value"
).dropna(subset=["revenue", "cost_of_revenue", "gross_profit"])
np.testing.assert_allclose(
    annual_identity.revenue - annual_identity.cost_of_revenue,
    annual_identity.gross_profit, rtol=2e-4, atol=2.0,
)

## Create the clean DCF fact ledger

This file keeps all validated NKE, DECK, and ONON facts, including older comparative periods. It is an audit ledger, not the final model-history selection. The currency remains as reported: USD for NKE and DECK, CHF for ONON, and shares for Nike share-count metrics.

In [ ]:
facts = raw_facts.loc[raw_facts.ticker.isin({"NKE", "DECK", "ONON"})].copy()
facts = facts.sort_values(fact_key).reset_index(drop=True)

expected_currency = {"NKE": {"USD", "shares"}, "DECK": {"USD"}, "ONON": {"CHF"}}
assert facts.groupby("ticker").currency.agg(set).to_dict() == expected_currency

clean_facts_path = out / "clean_facts.csv"
facts.to_csv(clean_facts_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {clean_facts_path} ({len(facts):,} rows)")

## Create the Foot Locker context ledger

Foot Locker is intentionally excluded from the DCF operating schedules but preserved in its own file for retailer context and auditability.

In [ ]:
foot_locker_facts = raw_facts.loc[raw_facts.ticker.eq("FL")].copy()
foot_locker_facts = foot_locker_facts.sort_values(fact_key).reset_index(drop=True)
assert not foot_locker_facts.empty
assert foot_locker_facts.currency.eq("USD").all()

foot_locker_path = out / "foot_locker_financial_facts.csv"
foot_locker_facts.to_csv(foot_locker_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {foot_locker_path} ({len(foot_locker_facts):,} rows)")

## Select the annual model history

The latest three annual financial periods are selected independently for each company. This choice is manual model policy. It yields NKE and DECK fiscal 2024–2026 and ONON calendar 2023–2025. Older ONON comparatives remain in `clean_facts.csv` but do not enter the three-year model schedule.

In [ ]:
annual_dates = (
    facts.loc[facts.duration.eq("annual"), ["ticker", "period_end"]]
    .drop_duplicates()
    .sort_values(["ticker", "period_end"])
    .groupby("ticker", group_keys=False)
    .tail(3)
)
expected_periods = {
    "NKE": {"2024-05-31", "2025-05-31", "2026-05-31"},
    "DECK": {"2024-03-31", "2025-03-31", "2026-03-31"},
    "ONON": {"2023-12-31", "2024-12-31", "2025-12-31"},
}
actual_periods = annual_dates.assign(
    period_end=annual_dates.period_end.dt.strftime("%Y-%m-%d")
).groupby("ticker").period_end.agg(set).to_dict()
assert actual_periods == expected_periods
annual_facts = facts.merge(annual_dates, on=["ticker", "period_end"])

## Define reusable financial calculations

Reported facts are pivoted into one row per company and reporting date. If operating income is unavailable, it is manually approximated as gross profit less selling, general, and administrative expense and labeled as such.

Observed tax rate is `income tax expense / pretax income` only when pretax income is positive. Cash flow after capital expenditures is `cash from operations - capital expenditures`.

In [ ]:
def wide_metrics(frame):
    return (
        frame.pivot(index=["ticker", "period_end"], columns="metric", values="value")
        .rename_axis(columns=None)
        .reset_index()
    )


def operating_drivers(frame):
    result = frame.copy()
    missing_ebit = result["operating_income"].isna()
    result["operating_income_basis"] = np.where(
        missing_ebit, "gross_profit_less_sga", "reported"
    )
    result.loc[missing_ebit, "operating_income"] = (
        result.loc[missing_ebit, "gross_profit"]
        - result.loc[missing_ebit, "selling_general_admin"]
    )
    return result

In [ ]:
def add_operating_ratios(frame):
    result = frame.copy()
    positive_revenue = result["revenue"].where(result.revenue.gt(0))
    for name, numerator in {
        "gross_margin": "gross_profit",
        "operating_margin": "operating_income",
        "da_to_revenue": "depreciation_amortization",
        "capex_to_revenue": "capital_expenditures",
    }.items():
        result[name] = result[numerator] / positive_revenue
    result["tax_rate_observed"] = (
        result.income_tax_expense / result.pretax_income.where(result.pretax_income.gt(0))
    )
    result["cash_flow_after_capex"] = (
        result.cash_from_operations - result.capital_expenditures
    )
    return result

## Build the annual operating schedule

The annual schedule retains one source URL for each company-period row. Metric-level line provenance remains available in `clean_facts.csv`.

In [ ]:
annual = wide_metrics(annual_facts.loc[annual_facts.duration.eq("annual")])
annual = add_operating_ratios(operating_drivers(annual))
annual["currency"] = annual.ticker.map({"NKE": "USD", "DECK": "USD", "ONON": "CHF"})
annual["duration"] = "annual"

annual_urls = (
    annual_facts.loc[annual_facts.duration.eq("annual")]
    .groupby(["ticker", "period_end"])
    .source_url.first()
    .rename("source_url")
    .reset_index()
)
annual = annual.merge(annual_urls, on=["ticker", "period_end"], validate="one_to_one")

## Build balance-sheet calculations

Operating net working capital is manually defined as receivables plus inventory plus other current assets, less accounts payable and other current liabilities. It is a proxy and not a filing-reported subtotal.

Debt requires both current and noncurrent reported debt values. If either component is absent, total and net debt remain missing. Share classes are combined only when both are reported.

In [ ]:
balance_facts = facts.loc[facts.duration.eq("instant")]
balance = wide_metrics(balance_facts)
balance["currency"] = balance.ticker.map({"NKE": "USD", "DECK": "USD", "ONON": "CHF"})

nwc_assets = balance[["accounts_receivable", "inventory", "other_current_assets"]]
nwc_liabilities = balance[["accounts_payable", "other_current_liabilities"]]
balance["operating_nwc_proxy"] = (
    nwc_assets.sum(axis=1, min_count=3) - nwc_liabilities.sum(axis=1, min_count=2)
)
balance["total_debt_reported"] = balance[["current_debt", "noncurrent_debt"]].sum(
    axis=1, min_count=2
)
balance["net_debt_reported"] = (
    balance.total_debt_reported - balance.cash_and_equivalents - balance.short_term_investments
)
balance["class_a_plus_b_shares"] = balance[[
    "class_a_shares_outstanding", "class_b_shares_outstanding"
]].sum(axis=1, min_count=2)

In [ ]:
balance_urls = balance_facts.groupby(["ticker", "period_end"]).source_url.first()
balance["source_url"] = balance_urls.reindex(
    pd.MultiIndex.from_frame(balance[["ticker", "period_end"]])
).to_numpy()
balance = balance.sort_values(["ticker", "period_end"]).reset_index(drop=True)

## Add working-capital changes and the FCFF proxy

Annual working-capital changes use consecutive fiscal year-end proxy balances. The FCFF proxy is:

`observed-tax NOPAT + depreciation and amortization - capital expenditures - change in operating NWC`

This is a manually defined modeling proxy. It is not presented as filing-reported free cash flow and should not replace a forecast working-capital analysis.

In [ ]:
fiscal_month = {"NKE": 5, "DECK": 3, "ONON": 12}
annual_balances = balance.loc[
    balance.period_end.dt.month.eq(balance.ticker.map(fiscal_month))
].copy()
annual_balances["delta_operating_nwc_proxy"] = (
    annual_balances.groupby("ticker").operating_nwc_proxy.diff()
)
annual = annual.merge(
    annual_balances[["ticker", "period_end", "operating_nwc_proxy", "delta_operating_nwc_proxy"]],
    on=["ticker", "period_end"], how="left", validate="one_to_one",
)
annual["nopat_observed_tax"] = annual.operating_income * (1 - annual.tax_rate_observed)
annual["fcff_proxy_observed_tax_nwc"] = (
    annual.nopat_observed_tax + annual.depreciation_amortization
    - annual.capital_expenditures - annual.delta_operating_nwc_proxy
)

## Create the native-currency annual history file

This file preserves reported currencies. ONON remains in CHF here so the source values and the later FX translation can be audited separately.

In [ ]:
annual = annual.sort_values(["ticker", "period_end"]).reset_index(drop=True)
assert len(annual) == 9
assert annual.revenue.gt(0).all()
assert annual.gross_margin.between(0, 1).all()
assert annual.operating_margin.between(-1, 1).all()
np.testing.assert_allclose(
    annual.revenue - annual.cost_of_revenue,
    annual.gross_profit, rtol=2e-4, atol=2.0,
)

annual_path = out / "historical_annual.csv"
annual.to_csv(annual_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {annual_path} ({len(annual):,} rows)")

## Build latest-twelve-month flows

The roll-forward formulas are manually specified by issuer because the available interim periods differ. Only flow metrics are rolled forward. Missing components remain missing rather than being treated as zero.

In [ ]:
flows = facts.loc[facts.duration.ne("instant")]
flow_metrics = sorted(flows.metric.unique())
latest_annual = annual.sort_values("period_end").groupby("ticker").tail(1)
ltm_rows = []

for ticker, duration in [("NKE", None), ("DECK", "3_months"), ("ONON", "6_months")]:
    base = latest_annual.loc[latest_annual.ticker.eq(ticker)].iloc[0]
    values = base.reindex(flow_metrics).copy()
    urls = [base.source_url]
    end = base.period_end
    method = "annual_only"
    if duration:
        parts = flows.loc[flows.ticker.eq(ticker) & flows.duration.eq(duration)]
        dates = sorted(parts.period_end.unique())
        assert len(dates) == 2, f"Expected two comparative periods for {ticker}"
        previous, current = [wide_metrics(parts.loc[parts.period_end.eq(date)]).iloc[0] for date in dates]
        values = values.add(current.reindex(flow_metrics)).sub(previous.reindex(flow_metrics))
        urls.extend(parts.loc[parts.period_end.isin(dates), "source_url"].drop_duplicates().tolist())
        end = dates[-1]
        method = f"annual_plus_current_{duration}_minus_prior_{duration}"
    ltm_rows.append({"ticker": ticker, "period_end": end, "currency": base.currency,
                     "duration": "ltm", "source_urls": " | ".join(urls),
                     "ltm_method": method, **values.to_dict()})

In [ ]:
ltm = add_operating_ratios(operating_drivers(pd.DataFrame(ltm_rows)))
ltm = ltm.sort_values("ticker").reset_index(drop=True)
expected_ltm_dates = {"NKE": "2026-05-31", "DECK": "2026-06-30", "ONON": "2026-06-30"}
actual_ltm_dates = ltm.assign(
    period_end=pd.to_datetime(ltm.period_end).dt.strftime("%Y-%m-%d")
).set_index("ticker").period_end.to_dict()
assert actual_ltm_dates == expected_ltm_dates
assert ltm[["revenue", "operating_income", "cash_from_operations", "capital_expenditures"]].notna().all().all()

## Create the native-currency LTM file

ONON remains in CHF in this file. The separate USD file created later uses the LTM reporting date's FX rate.

In [ ]:
ltm_path = out / "latest_ltm.csv"
ltm.to_csv(ltm_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {ltm_path} ({len(ltm):,} rows)")

## Create the latest balance-sheet file

The latest reported balance is selected independently for each company. The expected dates are manually tied to the raw filing set: May 2026 for Nike and June 2026 for Deckers and ONON.

In [ ]:
latest_balance = (
    balance.sort_values("period_end").groupby("ticker").tail(1)
    .sort_values("ticker").reset_index(drop=True)
)
expected_balance_dates = {"NKE": "2026-05-31", "DECK": "2026-06-30", "ONON": "2026-06-30"}
actual_balance_dates = latest_balance.assign(
    period_end=latest_balance.period_end.dt.strftime("%Y-%m-%d")
).set_index("ticker").period_end.to_dict()
assert actual_balance_dates == expected_balance_dates
assert latest_balance.cash_and_equivalents.gt(0).all()
assert 1_000_000_000 < latest_balance.set_index("ticker").loc["NKE", "class_a_plus_b_shares"] < 2_000_000_000

latest_balance_path = out / "latest_balance.csv"
latest_balance.to_csv(latest_balance_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {latest_balance_path} ({len(latest_balance):,} rows)")

## Clean and preserve the CHF/USD series

Yahoo Finance occasionally supplies rows without a closing value, including market holidays or incomplete observations. Such rows cannot support a conversion and are removed from the cleaned preprocessing copy. The raw file remains untouched.

In [ ]:
fx = fx_raw.copy()
numeric_fx = ["open", "high", "low", "close", "adj_close", "volume"]
fx[numeric_fx] = fx[numeric_fx].apply(pd.to_numeric, errors="coerce")
fx = fx.dropna(subset=["date", "adj_close"]).sort_values("date")
fx = fx.drop_duplicates("date", keep="last").reset_index(drop=True)
assert fx.date.is_monotonic_increasing and fx.date.is_unique
assert fx.adj_close.gt(0).all()
assert fx.date.min() <= pd.Timestamp("2021-01-01")
assert fx.date.max() >= pd.Timestamp("2026-06-30")

fx_path = out / "forex_chf_usd_clean.csv"
fx.to_csv(fx_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {fx_path} ({len(fx):,} rows)")

## Define the CHF-to-USD translation

`CHFUSD=X` is USD per CHF, so `USD amount = CHF amount × CHF/USD rate`.

For each reporting date, the latest valid adjusted close on or before that date is used. This backward-looking rule handles weekends and holidays without using future information. Ratios, share counts, dates, labels, and URLs are never multiplied by FX.

In [ ]:
def fx_rate_on_or_before(date_value):
    eligible = fx.loc[fx.date.le(pd.Timestamp(date_value))]
    if eligible.empty:
        raise ValueError(f"No CHF/USD observation on or before {date_value}")
    row = eligible.iloc[-1]
    return float(row.adj_close), row.date


def translate_onon_rows(frame, monetary_columns):
    result = frame.copy()
    result["source_currency"] = result["currency"]
    result["fx_rate_chf_usd"] = 1.0
    result["fx_rate_date"] = pd.to_datetime(result["period_end"])
    result["fx_method"] = "not_required_usd_source"
    onon_mask = result.ticker.eq("ONON")
    for index in result.index[onon_mask]:
        rate, rate_date = fx_rate_on_or_before(result.at[index, "period_end"])
        result.at[index, "fx_rate_chf_usd"] = rate
        result.at[index, "fx_rate_date"] = rate_date
        result.at[index, "fx_method"] = "latest_valid_adj_close_on_or_before_period_end"
        for column in monetary_columns:
            if column in result and pd.notna(result.at[index, column]):
                result.at[index, column] *= rate
    result["currency"] = "USD"
    return result

## Create the USD annual history file

The monetary-column list is manually defined to prevent accidental conversion of margins, tax rates, share counts, or identifiers. NKE and DECK values are already USD and remain numerically unchanged.

In [ ]:
annual_monetary = [
    "capital_expenditures", "cash_from_operations", "change_in_working_capital",
    "cost_of_revenue", "depreciation_amortization", "dividends_paid", "gross_profit",
    "income_tax_expense", "interest_paid", "net_income", "operating_income",
    "pretax_income", "revenue", "selling_general_admin", "stock_based_compensation",
    "cash_flow_after_capex", "operating_nwc_proxy", "delta_operating_nwc_proxy",
    "nopat_observed_tax", "fcff_proxy_observed_tax_nwc",
]
annual_usd = translate_onon_rows(annual, annual_monetary)

onon_native = annual.loc[annual.ticker.eq("ONON"), "revenue"].to_numpy()
onon_rates = annual_usd.loc[annual_usd.ticker.eq("ONON"), "fx_rate_chf_usd"].to_numpy()
onon_usd = annual_usd.loc[annual_usd.ticker.eq("ONON"), "revenue"].to_numpy()
np.testing.assert_allclose(onon_usd, onon_native * onon_rates)

annual_usd_path = out / "historical_annual_usd.csv"
annual_usd.to_csv(annual_usd_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {annual_usd_path} ({len(annual_usd):,} rows)")

## Create the USD LTM file

The LTM translation uses the reporting date rate for every ONON monetary flow. This is an explicit comparability choice; an accounting translation would normally use average rates for income-statement and cash-flow items.

In [ ]:
ltm_monetary = [column for column in annual_monetary if column in ltm.columns]
ltm_usd = translate_onon_rows(ltm, ltm_monetary)

onon_ltm_native = ltm.loc[ltm.ticker.eq("ONON"), "revenue"].iloc[0]
onon_ltm_rate = ltm_usd.loc[ltm_usd.ticker.eq("ONON"), "fx_rate_chf_usd"].iloc[0]
onon_ltm_usd = ltm_usd.loc[ltm_usd.ticker.eq("ONON"), "revenue"].iloc[0]
assert np.isclose(onon_ltm_usd, onon_ltm_native * onon_ltm_rate)

ltm_usd_path = out / "latest_ltm_usd.csv"
ltm_usd.to_csv(ltm_usd_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {ltm_usd_path} ({len(ltm_usd):,} rows)")

## Create the USD balance-sheet file

Balance-sheet monetary values are translated at the reporting-date rate. Share counts remain unconverted.

In [ ]:
balance_nonmonetary = {
    "ticker", "period_end", "currency", "source_url",
    "class_a_shares_outstanding", "class_b_shares_outstanding", "class_a_plus_b_shares",
}
balance_monetary = [
    column for column in latest_balance.select_dtypes(include="number").columns
    if column not in balance_nonmonetary
]
latest_balance_usd = translate_onon_rows(latest_balance, balance_monetary)

assert latest_balance_usd.loc[
    latest_balance_usd.ticker.eq("ONON"), "class_a_plus_b_shares"
].equals(latest_balance.loc[latest_balance.ticker.eq("ONON"), "class_a_plus_b_shares"])

balance_usd_path = out / "latest_balance_usd.csv"
latest_balance_usd.to_csv(balance_usd_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {balance_usd_path} ({len(latest_balance_usd):,} rows)")

## Create the WACC reference file

These externally sourced values are copied from the raw reference table after URL, date, and unit checks. No WACC is calculated here, and the industry beta is not automatically relevered.

In [ ]:
assert wacc.source_url.str.startswith("https://").all()
assert wacc.observation_date.notna().all()
assert wacc.loc[wacc.unit.eq("fraction"), "value"].between(0, 1).all()

wacc_path = out / "wacc_reference.csv"
wacc.to_csv(wacc_path, index=False)
print(f"Wrote {wacc_path} ({len(wacc):,} rows)")

## Create the market snapshot file

The raw peer snapshot has no observation date, so it is manually marked unusable for a current price target. Its `price_usd` column is already USD, including ONON's NYSE-listed price, and is deliberately not FX-converted.

In [ ]:
assert market.observation_date.isna().all()
assert market.loc[market.ticker.eq("ONON"), "price_usd"].notna().all()
market = market.copy()
market["usable_for_current_price_target"] = False

market_path = out / "market_snapshot_undated.csv"
market.to_csv(market_path, index=False)
print(f"Wrote {market_path} ({len(market):,} rows)")

## Define SEC Markdown parsing helpers for the channel case study

These helpers read the saved raw filing Markdown. Row labels are matched literally, and every output keeps the SEC URL and Markdown line number. This is deterministic extraction, not fuzzy or model-generated matching.

In [ ]:
def source_url(lines):
    return next(line.split(": ", 1)[1] for line in lines if line.startswith("source: "))


def markdown_cells(line):
    return [cell.strip() for cell in line.strip().strip("|").split("|")]


def filing_number(cell):
    match = re.search(r"-?\d[\d,]*(?:\.\d+)?", cell)
    if not match:
        raise ValueError(f"Expected a numeric filing cell: {cell!r}")
    return float(match.group().replace(",", ""))


def first_filing_row(lines, label, start=0):
    for line_number, line in enumerate(lines[start:], start + 1):
        if line.startswith(f"| {label} |") or line.startswith(f"| {label} ("):
            return line_number, markdown_cells(line)[1:]
    raise ValueError(f"No filing row found for {label}")

## Manually selected case-study filings

The source files and comparative-column positions below are manually chosen from the raw filing archive. Nike uses filings that collectively cover fiscal 2019–2026. Deckers uses each annual filing's current-year HOKA channel table. ONON uses its 2023 and 2025 annual filings to cover 2021–2025. Foot Locker uses all saved annual filings and the first matching Nike-purchase disclosure.

In [ ]:
nike_reports = [
    (raw / "sec_filings/NKE/10-K_2020-05-31_filed_2020-07-24.md", [(2020, 0), (2019, 2)]),
    (raw / "sec_filings/NKE/10-K_2023-05-31_filed_2023-07-20.md", [(2023, 0), (2022, 1), (2021, 4)]),
    (raw / "sec_filings/NKE/10-K_2026-05-31_filed_2026-07-15.md", [(2026, 0), (2025, 1), (2024, 4)]),
]
deck_reports = [
    raw / f"sec_filings/DECK/10-K_{year}-03-31_filed_{filed}.md"
    for year, filed in [
        (2019, "2019-05-30"), (2020, "2020-06-01"), (2021, "2021-05-28"),
        (2022, "2022-05-27"), (2023, "2023-05-26"), (2024, "2024-05-24"),
        (2025, "2025-05-23"), (2026, "2026-05-22"),
    ]
]
onon_reports = [
    (raw / "sec_filings/ONON/20-F_2023-12-31_filed_2024-03-12.md", [(2021, 2), (2022, 1)]),
    (raw / "sec_filings/ONON/20-F_2025-12-31_filed_2026-03-03.md", [(2023, 2), (2024, 1), (2025, 0)]),
]
foot_locker_reports = sorted((raw / "sec_filings/FL").glob("10-K_*.md"))

## Build Nike's channel history

Nike Brand total includes Global Brand Divisions in addition to wholesale and Nike Direct. The source identity is checked with all three components. The saved Direct share intentionally uses only wholesale plus Direct in its denominator.

In [ ]:
nike_rows = []
for path, years in nike_reports:
    lines = path.read_text(encoding="utf-8").splitlines()
    marker = next(i for i, line in enumerate(lines) if "Supplemental NIKE Brand Revenues Details:" in line)
    wholesale_line, wholesale = first_filing_row(lines, "Sales to Wholesale Customers", marker)
    direct_line, direct = first_filing_row(lines, "Sales through NIKE Direct", marker)
    _, divisions = first_filing_row(lines, "Global Brand Divisions (2)", marker)
    _, brand = first_filing_row(lines, "TOTAL NIKE BRAND REVENUES", marker)
    margin_label = "Gross margin (1)" if path.name.startswith("10-K_2020") else "Gross margin"
    margin_line, margin = first_filing_row(lines, margin_label)
    margin_values = [filing_number(cell) for cell in margin if re.fullmatch(r"\d{2}\.\d\s*%?", cell)]
    assert len(margin_values) >= len(years)
    for offset, (year, cell_index) in enumerate(years):
        w, d, g, total = [filing_number(row[cell_index]) for row in (wholesale, direct, divisions, brand)]
        assert abs(w + d + g - total) < 0.01
        nike_rows.append({
            "company": "Nike", "fiscal_year": year, "fiscal_year_end": f"{year}-05-31",
            "currency": "USD", "unit": "millions", "nike_brand_wholesale": w,
            "nike_brand_direct": d, "nike_brand_total": total,
            "direct_share_of_wholesale_plus_direct_pct": round(100 * d / (w + d), 2),
            "nike_inc_gross_margin_pct": margin_values[offset], "source_url": source_url(lines),
            "wholesale_md_line": wholesale_line, "direct_md_line": direct_line,
            "gross_margin_md_line": margin_line,
        })
nike_history = pd.DataFrame(nike_rows).sort_values("fiscal_year").reset_index(drop=True)

## Create the Nike channel-history file

The output keeps reported USD millions and source line references. No CHF conversion applies.

In [ ]:
nike_case_path = case_out / "nike_channel_history.csv"
nike_history.to_csv(nike_case_path, index=False)
print(f"Wrote {nike_case_path} ({len(nike_history):,} rows)")

## Build the peer channel history

Deckers filing tables report thousands and are divided by 1,000 to produce USD millions. ONON channel tables report CHF millions directly. The native currencies are retained because channel-growth comparisons use within-company percentage changes, not cross-currency subtraction.

In [ ]:
peer_rows = []
for path in deck_reports:
    lines = path.read_text(encoding="utf-8").splitlines()
    marker = next(i for i, line in enumerate(lines) if line.startswith("| HOKA brand |"))
    wholesale_line, wholesale = first_filing_row(lines, "Wholesale", marker)
    direct_line, direct = first_filing_row(lines, "Direct-to-Consumer", marker)
    _, total = first_filing_row(lines, "Total", marker)
    year = int(path.name.split("_")[1][:4])
    w, d, total_value = [filing_number(row[0]) / 1_000 for row in (wholesale, direct, total)]
    assert abs(w + d - total_value) < 0.01
    peer_rows.append({
        "company": "HOKA / Deckers", "fiscal_year": year, "fiscal_year_end": f"{year}-03-31",
        "currency": "USD", "unit": "millions", "wholesale_revenue": w,
        "direct_revenue": d, "total_revenue": total_value,
        "direct_share_pct": round(100 * d / total_value, 2),
        "scope": "HOKA brand; all products", "source_url": source_url(lines),
        "wholesale_md_line": wholesale_line, "direct_md_line": direct_line,
    })

In [ ]:
for path, years in onon_reports:
    lines = path.read_text(encoding="utf-8").splitlines()
    wholesale_line, wholesale = first_filing_row(lines, "Wholesale")
    direct_line, direct = first_filing_row(lines, "Direct-to-Consumer")
    _, total = first_filing_row(lines, "Net sales")
    for year, cell_index in years:
        w, d, total_value = [filing_number(row[cell_index]) for row in (wholesale, direct, total)]
        assert abs(w + d - total_value) < 0.11
        peer_rows.append({
            "company": "On Holding", "fiscal_year": year, "fiscal_year_end": f"{year}-12-31",
            "currency": "CHF", "unit": "millions", "wholesale_revenue": w,
            "direct_revenue": d, "total_revenue": total_value,
            "direct_share_pct": round(100 * d / total_value, 2),
            "scope": "On consolidated; all products", "source_url": source_url(lines),
            "wholesale_md_line": wholesale_line, "direct_md_line": direct_line,
        })
peer_history = pd.DataFrame(peer_rows).sort_values(["company", "fiscal_year"]).reset_index(drop=True)

## Create the peer channel-history file

The currency column prevents accidental aggregation across USD and CHF. Company-level identities are validated before the file is written.

In [ ]:
np.testing.assert_allclose(
    peer_history.wholesale_revenue + peer_history.direct_revenue,
    peer_history.total_revenue, rtol=0, atol=0.11,
)
peer_case_path = case_out / "peer_channel_history.csv"
peer_history.to_csv(peer_case_path, index=False)
print(f"Wrote {peer_case_path} ({len(peer_history):,} rows)")

## Build Foot Locker's Nike-purchase concentration history

The disclosure pattern and the 2019 lower bound are manually specified. The metric is Foot Locker's share of merchandise purchases from Nike; it is not Nike revenue, footwear sell-through, or market share.

In [ ]:
purchase_pattern = re.compile(
    r"Approximately\s+(\d+)\s*(?:%|percent)\s+of all merchandise purchased in "
    r"(\d{4}) was purchased from one supplier\s*[—-]\s*Nike",
    re.IGNORECASE,
)
foot_locker_rows = []
for path in foot_locker_reports:
    lines = path.read_text(encoding="utf-8").splitlines()
    for line_number, line in enumerate(lines, 1):
        match = purchase_pattern.search(line)
        if match:
            year = int(match.group(2))
            if year >= 2019:
                foot_locker_rows.append({
                    "purchase_year": year,
                    "nike_share_of_foot_locker_purchases_pct": int(match.group(1)),
                    "metric": "Share of all merchandise purchases from Nike",
                    "source_url": source_url(lines), "md_line": line_number,
                })
            break
foot_locker_history = pd.DataFrame(foot_locker_rows).sort_values("purchase_year").reset_index(drop=True)

## Create the Foot Locker concentration file

Every observation retains its SEC URL and exact Markdown line number.

In [ ]:
assert foot_locker_history.purchase_year.between(2019, 2026).all()
assert foot_locker_history.source_url.str.startswith("https://www.sec.gov/Archives/").all()

foot_locker_case_path = case_out / "foot_locker_nike_concentration.csv"
foot_locker_history.to_csv(foot_locker_case_path, index=False)
print(f"Wrote {foot_locker_case_path} ({len(foot_locker_history):,} rows)")

## Build the daily equity-price history

The case-study price chart uses Yahoo Finance adjusted closes for Nike, Deckers, and On.
Adjusted close incorporates splits and distributions reported by the vendor, which is preferable
to raw close for a long comparison. Rows outside January 1, 2019 through September 15, 2026 are
removed. Each observation retains its provider URL.

No pre-listing On prices are created. Missing trading dates remain missing and are handled only
when the research notebook draws each observed series.

In [ ]:
price_specs = {
    'Nike': ('NKE', raw / 'prices/nke.csv'),
    'Deckers': ('DECK', raw / 'prices/deck.csv'),
    'On': ('ONON', raw / 'prices/onon.csv'),
}
price_frames = []
for company, (ticker, path) in price_specs.items():
    frame = pd.read_csv(path)
    frame['date'] = pd.to_datetime(frame.date)
    frame = frame.loc[frame.date.between('2019-01-01', '2026-09-15')].copy()
    assert frame.ticker.eq(ticker).all()
    assert frame.adj_close.dropna().gt(0).all()
    frame['company'] = company
    frame['source_url'] = f'https://query2.finance.yahoo.com/v8/finance/chart/{ticker}'
    price_frames.append(frame[['date', 'company', 'ticker', 'adj_close', 'source_url']])

equity_prices = (pd.concat(price_frames, ignore_index=True)
                 .dropna(subset=['adj_close'])
                 .sort_values(['company', 'date'])
                 .reset_index(drop=True))
assert not equity_prices.duplicated(['ticker', 'date']).any()
assert equity_prices.groupby('ticker').adj_close.size().gt(1000).all()
assert equity_prices.loc[equity_prices.ticker.eq('ONON'), 'date'].min().year == 2021

price_case_path = case_out / 'equity_price_history.csv'
equity_prices.to_csv(price_case_path, index=False)
print(f'Wrote {price_case_path} ({len(equity_prices):,} rows)')

## Validate the complete rebuilt directory

The checks below confirm the exact expected file set, row counts, currencies, period selections, accounting identities, FX arithmetic, and case-study identities. Any failed assertion stops the notebook before the manifest is created.

In [ ]:
expected_top_level = {
    "clean_facts.csv", "foot_locker_financial_facts.csv", "historical_annual.csv",
    "historical_annual_usd.csv", "latest_ltm.csv", "latest_ltm_usd.csv",
    "latest_balance.csv", "latest_balance_usd.csv", "forex_chf_usd_clean.csv",
    "wacc_reference.csv", "market_snapshot_undated.csv",
}
expected_case_files = {
    "nike_channel_history.csv", "peer_channel_history.csv",
    "foot_locker_nike_concentration.csv", "equity_price_history.csv",
}
assert {path.name for path in out.glob("*.csv")} == expected_top_level
assert {path.name for path in case_out.glob("*.csv")} == expected_case_files
assert len(facts) == 685 and len(foot_locker_facts) == 71
assert len(annual) == 9 and len(ltm) == 3 and len(latest_balance) == 3
assert annual_usd.currency.eq("USD").all()
assert ltm_usd.currency.eq("USD").all()
assert latest_balance_usd.currency.eq("USD").all()

In [ ]:
non_onon = annual.ticker.ne("ONON")
for column in annual_monetary:
    if column in annual and column in annual_usd:
        np.testing.assert_allclose(
            annual.loc[non_onon, column], annual_usd.loc[non_onon, column],
            equal_nan=True,
        )

nike_direct_share = 100 * nike_history.nike_brand_direct / (
    nike_history.nike_brand_wholesale + nike_history.nike_brand_direct
)
np.testing.assert_allclose(
    nike_direct_share,
    nike_history.direct_share_of_wholesale_plus_direct_pct,
    atol=0.01,
)
assert market.loc[market.ticker.eq("ONON"), "price_usd"].iloc[0] == 27.47
print("All preprocessing validations passed.")

## Create the preprocessing manifest

The manifest is generated from the files that now exist. Its descriptive source and currency-policy fields are manually authored so downstream users can see which outputs are native currency, translated USD, or nonfinancial reference data.

In [ ]:
manifest_policy = {
    "clean_facts.csv": ("raw SEC DCF facts", "reported currency"),
    "foot_locker_financial_facts.csv": ("raw SEC DCF facts", "USD"),
    "historical_annual.csv": ("raw SEC DCF facts", "reported currency"),
    "historical_annual_usd.csv": ("historical_annual plus raw CHF/USD", "USD"),
    "latest_ltm.csv": ("raw SEC DCF facts", "reported currency"),
    "latest_ltm_usd.csv": ("latest_ltm plus raw CHF/USD", "USD"),
    "latest_balance.csv": ("raw SEC DCF facts", "reported currency"),
    "latest_balance_usd.csv": ("latest_balance plus raw CHF/USD", "USD"),
    "forex_chf_usd_clean.csv": ("raw Yahoo Finance CHFUSD=X", "USD per CHF"),
    "wacc_reference.csv": ("raw market input references", "mixed units"),
    "market_snapshot_undated.csv": ("raw peer snapshot", "USD prices and market caps"),
}
manifest_rows = []
for name, (source_scope, currency_policy) in manifest_policy.items():
    path = out / name
    manifest_rows.append({"file": str(path.relative_to(root)), "rows": len(pd.read_csv(path)),
                          "source_scope": source_scope, "currency_policy": currency_policy})
for path in sorted(case_out.glob("*.csv")):
    if path.name == 'equity_price_history.csv':
        source_scope, currency_policy = 'raw Yahoo Finance adjusted closes', 'USD per share'
    else:
        source_scope, currency_policy = 'raw SEC filing Markdown', 'reported currency'
    manifest_rows.append({"file": str(path.relative_to(root)), "rows": len(pd.read_csv(path)),
                          "source_scope": source_scope, "currency_policy": currency_policy})

manifest = pd.DataFrame(manifest_rows).sort_values("file").reset_index(drop=True)
manifest_path = out / "preprocessing_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Wrote {manifest_path} ({len(manifest):,} rows)")

## Rebuild complete

The output directory is now fully reconstructed from `data/raw`. Review `preprocessing_manifest.csv` for the generated file inventory. Native-currency and USD-converted financial schedules are both retained so every ONON translation can be traced to a dated CHF/USD observation.